In [1]:
import pandas as pd


In [3]:
df = pd.read_csv('../data/combined.csv')


In [4]:
df

,ResponseId,Age,Gender,shopping_freq,cookie,cup,sneakers,headphones,perfume,treated
0,R_57yhLhJSJEty4Sw,2,1,4,1.5,3.0,49.99,40.0,60,0
1,R_35TlKdJzxulrBFx,2,2,3,10.0,20.0,30.00,50.0,80,0
2,R_5MgxYxUD8U3eTLj,2,2,2,2.0,8.0,60.00,90.0,80,0
3,R_6a9QX5xuLBC8LEJ,2,2,4,3.0,3.0,70.00,50.0,50,0
4,R_1o6qNSuThZO9zoZ,2,2,3,7.0,7.0,200.00,158.0,35,0
...,...,...,...,...,...,...,...,...,...,...
59,R_1WsDuVxioyJhy49,2,2,3,20.0,30.0,200.00,150.0,80,1
60,R_5v1c2LwhVv5SiUV,2,1,4,4.0,20.0,100.00,70.0,120,1
61,R_3I3Hz9gi4i1hxVl,2,1,1,3.0,5.0,100.00,80.0,0,1
62,R_5rUsk5Z7fpdUmc5,2,1,1,5.0,60.0,200.00,200.0,250,1


# Randomization Checks

Check whether treatment arms are balanced and allocation proportions match the intended 20% per arm (1/5 for 5 arms).


In [11]:
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import chisquare
import numpy as np

# First, check what columns are available
print("Available columns in dataframe:")
print(df.columns.tolist())
print()

# Check allocation proportions across treatment arms
print("=== Treatment Arm Allocation ===\n")

# Use 'treated' column instead of 'treatment_arm'
treatment_col = 'treated'

if treatment_col not in df.columns:
    print(f"⚠ Column '{treatment_col}' not found in dataframe.")
else:
    treatment_counts = df[treatment_col].value_counts().sort_index()
    print("Treatment arm counts:")
    print(treatment_counts)

    total_n = len(df)
    print(f"\nTotal observations: {total_n}")

    # Expected proportion for each arm (1/2 = 0.5 for binary treatment)
    expected_prop = 0.5
    expected_count = total_n * expected_prop

    print(f"\nExpected count per arm (if equal allocation): {expected_count:.1f}")
    print(f"Expected proportion per arm: {expected_prop}")

    # Test each treatment arm against expected proportion
    print("\n=== Individual Proportions z-tests ===")
    print("H0: proportion = 0.5 for each arm (two-sided)\n")

    for arm in treatment_counts.index:
        count = treatment_counts[arm]
        obs_prop = count / total_n
        z_stat, p_value = proportions_ztest(count, total_n, expected_prop)
        diff = obs_prop - expected_prop
        print(f"Arm {arm}: n={count}, prop={obs_prop:.4f}, diff={diff:+.4f}, z={z_stat:+.3f}, p={p_value:.4f}")

    # Overall chi-square goodness-of-fit test
    print("\n=== Overall Chi-Square Goodness-of-Fit Test ===")
    print("H0: all arms have equal allocation (0.5 each)\n")

    # Calculate expected frequencies to ensure they sum to the same total as observed
    total_observed = treatment_counts.values.sum()
    n_arms = len(treatment_counts)
    expected_freq = np.full(n_arms, total_observed / n_arms)
    
    chi2_stat, chi2_p = chisquare(treatment_counts.values, f_exp=expected_freq)
    print(f"Chi-square statistic: {chi2_stat:.4f}")
    print(f"Degrees of freedom: {n_arms - 1}")
    print(f"P-value: {chi2_p:.4f}")

    # Interpretation
    print("\n=== Interpretation ===")
    if chi2_p > 0.05:
        print("✓ No evidence of imbalanced allocation (p > 0.05).")
        print("  The observed differences are consistent with random variation.")
    else:
        print("⚠ Evidence of imbalanced allocation (p < 0.05).")
        print("  Treatment arms differ significantly from equal allocation.")

Available columns in dataframe:
['ResponseId', 'Age', 'Gender', 'shopping_freq', 'cookie', 'cup', 'sneakers', 'headphones', 'perfume', 'treated']

=== Treatment Arm Allocation ===

Treatment arm counts:
treated
0    34
1    30
Name: count, dtype: int64

Total observations: 64

Expected count per arm (if equal allocation): 32.0
Expected proportion per arm: 0.5

=== Individual Proportions z-tests ===
H0: proportion = 0.5 for each arm (two-sided)

Arm 0: n=34, prop=0.5312, diff=+0.0312, z=+0.501, p=0.6164
Arm 1: n=30, prop=0.4688, diff=-0.0312, z=-0.501, p=0.6164

=== Overall Chi-Square Goodness-of-Fit Test ===
H0: all arms have equal allocation (0.5 each)

Chi-square statistic: 0.2500
Degrees of freedom: 1
P-value: 0.6171

=== Interpretation ===
✓ No evidence of imbalanced allocation (p > 0.05).
  The observed differences are consistent with random variation.


In [ ]:
# Check balance on baseline (pre-treatment) covariates
import statsmodels.formula.api as smf

print("\n" + "="*60)
print("=== Covariate Balance Check (Baseline) ===")
print("="*60)

# Baseline covariates (pre-treatment characteristics)
baseline_vars = ['Age', 'Gender', 'shopping_freq']

# Filter to only columns that exist
baseline_vars = [v for v in baseline_vars if v in df.columns]

if not baseline_vars:
    print("No suitable baseline covariates found for balance check.")
else:
    print("Testing whether baseline covariates differ by treatment arm...")
    print("(Using OLS with treated indicator)\n")

    for var in baseline_vars:
        if var not in df.columns:
            print(f"Skipping {var}: not in dataset")
            continue
        
        # Drop missing values
        df_clean = df[[var, 'treated']].dropna()
        
        if len(df_clean) < 5:
            print(f"Skipping {var}: too few observations after dropping NaN")
            continue
        
        # Convert to numeric (handle categorical if needed)
        try:
            df_clean[var] = pd.to_numeric(df_clean[var], errors='coerce')
            df_clean = df_clean.dropna(subset=[var])
        except Exception:
            pass
        
        # Fit OLS: baseline_var ~ treated
        model = smf.ols(f'{var} ~ treated', data=df_clean).fit()
        
        # Extract results
        coef = model.params['treated']
        se = model.bse['treated']
        t_stat = model.tvalues['treated']
        p_val = model.pvalues['treated']
        
        # Group means
        mean_control = df_clean[df_clean['treated'] == 0][var].mean()
        mean_treat = df_clean[df_clean['treated'] == 1][var].mean()
        
        print(f"{var}:")
        print(f"  Mean (control):   {mean_control:.4f}")
        print(f"  Mean (treatment): {mean_treat:.4f}")
        print(f"  Difference:       {coef:+.4f} (SE={se:.4f})")
        print(f"  t={t_stat:+.3f}, p={p_val:.4f}")
        
        if p_val < 0.05:
            print(f"  ⚠ Statistically significant difference (p < 0.05)")
        else:
            print(f"  ✓ No significant difference (p >= 0.05)")
        print()

print("="*60)
print("Summary: If all p-values >= 0.05, randomization likely worked.")
print("="*60)


=== Covariate Balance Check (Baseline) ===

Testing whether baseline covariates differ by treatment arm...
(Using OLS with any_treatment indicator)

Skipping racism_scores_pre_2mon: not in dataset
Skipping log_followers: not in dataset
Summary: If all p-values >= 0.05, randomization likely worked.
